## **ENTRENAMIENTO SAM 3 CON REFCOCOG**

En este fichero se evalúa el rendimiento del modelo SAM 3 tras haberlo entrenado usando el dataset RefCOCOg.

En este primer bloque de código se importan las librerías necesarias para la ejecución del fichero completo. Además, se importan las funciones de métricas y de medición de inferencia. Para ello se tuvo que añadir la raíz del proyecto al path de Python para que estos imports funcionen correctamente. Finalmente, se define la ruta al datset con el que entrenaremos los modelos.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from ultralytics import SAM
from ultralytics.models.sam import SAM3Predictor
from PIL import Image
from pycocotools import mask as maskUtils
from pycocotools.coco import COCO
import numpy as np
import os
import cv2
import pandas as pd
import time
import sys
import torch
import shutil
import random
import torch.nn as nn
import pickle


root_path = os.path.abspath('..')
if root_path not in sys.path:
    sys.path.append(root_path)

from utils.segmentation_quality_metrics import boundary_iou, hausdorff_95, compute_all_metrics
from utils.efficiency_metrics import measure_inference_refcocog, measure_inference_sam3_prompt_refcocog

dataset = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\refcocog"


En primer lugar, se definen las rutas al dataset y los parámetros del subconjunto que se va a usar: 1000 imágenes divididas en train, val y test (70, 15, 15).

La función split_dataset carga las referencias del fichero refs(umd).p y filtra únicamente las del split de entrenamiento original de RefCOCOg. A continuación selecciona aleatoriamente un subconjunto de 1000 imágenes con una semilla fija, lo divide en tres splits y copia las imágenes a sus carpetas correspondientes, generando y guardando también la máscara de cada imagen. Los ficheros se nombran por ann_id en lugar del nombre de imagen original, ya que una misma imagen puede contener múltiples referencias a objetos distintos y cada par imagen-máscara debe poder identificarse de forma inequívoca.

In [ ]:
DATASET_ROOT = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\refcocog"
IMAGES_DIR   = os.path.join(DATASET_ROOT, "train2014")
INSTANCES    = os.path.join(DATASET_ROOT, "instances.json")
SUBSET_SIZE  = 1000
TRAIN_RATIO  = 0.70
VAL_RATIO    = 0.15

def split_dataset():
    with open(os.path.join(DATASET_ROOT, "refs(umd).p"), "rb") as f:
        refs = pickle.load(f)
    coco = COCO(INSTANCES)

    random.seed(42)
    train_refs = [r for r in refs if r["split"] == "train"]
    random.shuffle(train_refs)
    subset = train_refs[:SUBSET_SIZE]

    n = len(subset)
    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)

    splits = {
        "train": subset[:n_train],
        "val":   subset[n_train:n_train + n_val],
        "test":  subset[n_train + n_val:]
    }

    for split, split_refs in splits.items():
        os.makedirs(os.path.join(DATASET_ROOT, "images", split), exist_ok=True)
        os.makedirs(os.path.join(DATASET_ROOT, "masks",  split), exist_ok=True)

        for ref in split_refs:
            ann = coco.loadAnns(ref["ann_id"])[0]
            img_info = coco.loadImgs(ref["image_id"])[0]

            src = os.path.join(IMAGES_DIR, img_info["file_name"])
            if not os.path.exists(src):
                continue

            ann_id = ref["ann_id"]
            dst_img  = os.path.join(DATASET_ROOT, "images", split, f"{ann_id}.jpg")
            dst_mask = os.path.join(DATASET_ROOT, "masks",  split, f"{ann_id}.png")

            shutil.copy(src, dst_img)

            mask = coco.annToMask(ann) * 255
            Image.fromarray(mask.astype(np.uint8)).save(dst_mask)

split_dataset()

loading annotations into memory...
Done (t=4.39s)
creating index...
index created!


En primer lugar, se habilita TF32 para las operaciones matriciales, ya que esto nos permite acelerar el entrenamiento en GPUs NVIDIA modernas con una pérdida de precisión mínima. Además, se define el dispositivo donde se ejecutará el entrenamiento y se comprueba la disponibilidad de CUDA.

A continuación, se define el dataset cargando en el constructor las rutas de las imágenes y máscaras del split correspondiente (train, val o test). También, se filtran aquellas muestras cuya máscara no existe o está vacía (sin píxeles activos), ya que no aportarían información útil al entrenamiento.

Por último, en el método __getitem__ se aplica el preprocesamiento de cada muestra. La imagen se redimensiona a 1008x1008 (múltiplo de 14, requerido por el ViT de SAM 3), se normaliza entre 0 y 1 y se convierte a tensor. Por otro lado, la máscara se redimensiona a 288x288 (resolución de salida del mask decoder de SAM 3) y se binariza. Finalmente, el punto central se calcula directamente a partir de los píxeles activos de la máscara en su resolución original y sus coordenadas se escalan proporcionalmente al tamaño de entrada del modelo. Además, se utiliza el centro de la imagen como valor por defecto si la máscara está vacía.

In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"

class SegDataset(Dataset):
    def __init__(self, dataset_path, split, img_size=1008):
        self.img_size   = img_size
        self.images_dir = os.path.join(dataset_path, "images", split)
        self.masks_dir  = os.path.join(dataset_path, "masks",  split)
        self.samples    = []

        for img_name in os.listdir(self.images_dir):
            if not img_name.endswith(".jpg"):
                continue
            name      = img_name.replace(".jpg", "")
            img_path  = os.path.join(self.images_dir, img_name)
            mask_path = os.path.join(self.masks_dir, name + ".png")
            if not os.path.exists(mask_path):
                continue
            gt = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if gt is None or (gt > 127).sum() == 0:
                continue
            self.samples.append((img_path, mask_path))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (self.img_size, self.img_size))
        image = torch.tensor(image).permute(2, 0, 1).float() / 255.0

        gt_full = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        orig_h, orig_w = gt_full.shape
        ys, xs = np.where(gt_full > 127)
        if len(xs) > 0:
            cx = float(xs.mean()) * (self.img_size / orig_w)
            cy = float(ys.mean()) * (self.img_size / orig_h)
        else:
            cx, cy = self.img_size / 2, self.img_size / 2

        mask = cv2.resize(gt_full, (288, 288))
        mask_tensor = torch.tensor((mask > 127).astype(np.float32)).unsqueeze(0)

        point = torch.tensor([[cx, cy]]).float()
        label = torch.tensor([1])
        return image, mask_tensor, point, label


Esta función contiene el bucle de entrenamiento. En primer lugar, se carga el modelo mediante el wrapper de Ultralytics SAM, del que se extrae el modelo interno con .model. Al igual que con los modelos anteriores, se congelan tanto el image encoder como el prompt encoder, de forma que durante el entrenamiento solo se actualizan los pesos del mask decoder.

Como optimizador se usa Adam con un learning rate de 1e-4, y se añade un scheduler que reduce el learning rate a la mitad cada 20 épocas, ayudando así a afinar el modelo conforme avanza el entrenamiento. Además, se usa BCEWithLogitsLoss como función de pérdida, ya que es adecuada para segmentación binaria.

Por último, al no poder usar predictor.set_image directamente, el image encoder se ejecuta manualmente mediante forward_image y _prepare_backbone_features, configurando explícitamente los tamaños de características esperados por el decoder. El resto del proceso es equivalente al de SAM 2, ya que el prompt encoder y el image encoder se ejecutan sin calcular gradientes, el mask decoder sí los calcula, la pérdida se promedia sobre el batch y los pesos se guardan en formato de diccionario con la clave "model".

In [ ]:
def train_sam(dataset_path, weights_path, output_weights, epochs=50, batch_size=4):
    sam3_wrapper = SAM(weights_path)
    sam3 = sam3_wrapper.model
    for param in sam3.parameters():
        param.data = param.data.to(device)
    for buffer in sam3.buffers():
        buffer.data = buffer.data.to(device)
    sam3.to(device)

    for param in sam3.image_encoder.parameters():
        param.requires_grad = False
    for param in sam3.sam_prompt_encoder.parameters():
        param.requires_grad = False

    optimizer = torch.optim.Adam(sam3.sam_mask_decoder.parameters(), lr=1e-4)
    scheduler  = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
    loss_fn   = nn.BCEWithLogitsLoss()

    train_dataset = SegDataset(dataset_path, "train")
    train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    predictor = SAM3Predictor(overrides={"model": weights_path, "task": "segment", "mode": "predict", "verbose": False})
    predictor.setup_model(sam3)

    predictor._bb_feat_sizes = [(288, 288), (144, 144), (72, 72)]
    
    sam3.train()

    for epoch in range(epochs):
        total_loss = 0
        for images, masks, points, labels in train_loader:
            masks  = masks.to(device)
            points = points.to(device)
            labels = labels.to(device)

            loss_batch = 0
            for i in range(images.shape[0]):

                with torch.no_grad():
                    image_tensor = images[i].unsqueeze(0).to(device)
                    backbone_out = sam3.forward_image(image_tensor)
                    _, vision_feats, _, _ = sam3._prepare_backbone_features(backbone_out)
                    #print(len(vision_feats), [v.shape for v in vision_feats])
                    if sam3.directly_add_no_mem_embed:
                        vision_feats[-1] = vision_feats[-1] + sam3.no_mem_embed
                    feats = [
                        feat.permute(1, 2, 0).reshape(1, -1, *feat_size)
                        for feat, feat_size in zip(vision_feats, predictor._bb_feat_sizes)
                    ]
                    features = {"image_embed": feats[-1], "high_res_feats": feats[:-1]}

                with torch.no_grad():
                    sparse_embeddings, dense_embeddings = sam3.sam_prompt_encoder(
                        points=(points[i].unsqueeze(0), labels[i].unsqueeze(0)),
                        boxes=None,
                        masks=None
                    )

                image_embedding = features["image_embed"]
                image_pe        = sam3.sam_prompt_encoder.get_dense_pe()
                high_res_features = features["high_res_feats"]

                low_res_masks, _, _, _ = sam3.sam_mask_decoder(
                    image_embeddings=image_embedding,
                    image_pe=image_pe,
                    sparse_prompt_embeddings=sparse_embeddings,
                    dense_prompt_embeddings=dense_embeddings,
                    multimask_output=False,
                    repeat_image=False,
                    high_res_features=high_res_features,
                )

                loss_batch += loss_fn(low_res_masks, masks[i].unsqueeze(0))

            loss = loss_batch / images.shape[0]
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        scheduler.step()
        print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")

    torch.save({"model": sam3.state_dict()}, output_weights)
    return output_weights


**SAM 3 ENTRENAMIENTO**

En primer lugar, se definen las rutas a los distintos recursos que se van a usar: el dataset, los pesos preentrenados y las rutas donde se guardarán los resultados y los pesos del modelo entrenado. En este caso, se procede a evaluar el rendimiento del modelo SAM 3.

Respecto a la función de evaluación, esta función evalúa el modelo fine-tuneado sobre el conjunto de test. Los pesos entrenados se cargan manualmente extrayendo el modelo interno del wrapper de Ultralytics y aplicando el state dict guardado durante el entrenamiento. Para cada referencia del split de test se carga la anotación individual desde la API de COCO, se genera su máscara con coco.annToMask y se descarta si está vacía. El prompt utilizado para la inferencia es la caja delimitadora de la anotación, igual que en la evaluación zero-sho, permitiendo así la comparación entre ambas. La máscara predicha se redimensiona al tamaño original de la imagen para que sea comparable con el ground truth, y se calculan todas las métricas. Al final se devuelve un diccionario con la media de cada métrica sobre todo el conjunto de test.

Finalmente, se lanza el entrenamiento midiendo el tiempo que tarda, y después se evalúa el modelo resultante, midiendo también su tiempo. Ambos tiempos se añaden al diccionario de resultados, que finalmente se guarda en un CSV. Si el fichero ya existe se añade una nueva línea sin sobreescribir los resultados anteriores, lo que permite ir acumulando los resultados de distintos experimentos en el mismo fichero.

In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"

dataset = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\refcocog"
weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam3.pt"
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"
output_weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_finetuned\\pesos_sam3_refcocog.pt"

def evaluate_model(dataset, weights_path):
    sam3_wrapper = SAM(weights)
    sam3 = sam3_wrapper.model
    for param in sam3.parameters():
        param.data = param.data.to(device)
    for buffer in sam3.buffers():
        buffer.data = buffer.data.to(device)
    sd = torch.load(weights_path)["model"]
    sam3.load_state_dict(sd)
    sam3.eval()

    with open(os.path.join(dataset, "refs(umd).p"), "rb") as f:
        refs = pickle.load(f)
    coco = COCO(os.path.join(dataset, "instances.json"))
    refs_test = [r for r in refs if r["split"] == "test"]

    iou_scores, precision_scores, recall_scores, f1_scores = [], [], [], []
    dice_scores, specificity_scores, f2_scores, pixel_accuracy_scores = [], [], [], []
    boundary_iou_scores, hausdorff_95_scores, latency_scores, vram_scores = [], [], [], []

    images_dir = os.path.join(dataset, "train2014")

    for ref in refs_test:
        ann = coco.loadAnns(ref["ann_id"])[0]
        img_info = coco.loadImgs(ref["image_id"])[0]
        img_path = os.path.join(images_dir, img_info["file_name"])

        gt_mask = coco.annToMask(ann).astype(bool)
        if gt_mask.sum() == 0:
            continue

        x, y, w, h = ann["bbox"]
        bbox = [x, y, x + w, y + h]

        results, latency, vram = measure_inference_refcocog(sam3_wrapper, img_path, bbox)

        if results is None or results[0].masks is None:
            continue
        scores = results[0].boxes.conf.cpu().numpy()
        if len(scores) == 0:
            continue

        H, W = gt_mask.shape
        best_idx = np.argmax(scores)
        pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        recall_scores.append(recall)
        f1_scores.append(f1)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    results = {
        "modelo": ["sam_3_refcocog"],
        "mean_iou": [np.mean(iou_scores)],
        "mean_f1": [np.mean(f1_scores)],
        "mean_recall": [np.mean(recall_scores)],
        "mean_precision": [np.mean(precision_scores)],
        "mean_dice": [np.mean(dice_scores)],
        "mean_specificity": [np.mean(specificity_scores)],
        "mean_f2": [np.mean(f2_scores)],
        "mean_pixel_accuracy": [np.mean(pixel_accuracy_scores)],
        "mean_boundary_iou": [np.mean(boundary_iou_scores)],
        "mean_hausdorff_95": [np.mean(hausdorff_95_scores)],
        "mean_latency_ms": [np.mean(latency_scores)],
        "mean_vram_mb": [np.mean(vram_scores)]
    }

    return results
         

start_train = time.time()
trained_weights = train_sam(dataset, weights, output_weights)
train_time = time.time() - start_train

start_eval = time.time()
results = evaluate_model(dataset, trained_weights)
eval_time = time.time() - start_eval

results["train_time_minutes"] = [round(train_time / 60, 2)]
results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"

if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)


c:\Users\DanielTalavera\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ultralytics 8.4.26  Python-3.10.11 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3090, 24575MiB)


C:\Users\DanielTalavera\AppData\Local\Temp\ipykernel_5200\2289603131.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load(weights_path)["model"]


WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must

**SAM 3 ENTRENAMIENTO CON PROMPT TEXTUAL**

En este bloque se evalúa una variante del modelo SAM 3 fine-tuneado sobre RefCOCOg, usando prompts textuales en lugar de puntos como prompts de inferencia. Se añade la ruta al fichero de referencias refs(umd).p, del que se extrae directamente la descripción en lenguaje natural asociada a cada referencia.

La función de evaluación carga los pesos fine-tuneados de la misma forma que en el bloque anterior. Para cada referencia del split de test se carga la anotación individual desde la API de COCO, se genera su máscara con coco.annToMask y se descarta si está vacía. La descripción textual se obtiene del campo sentences de cada referencia y se usa como prompt para la inferencia mediante Grounding DINO y SAM 3. Esto hace que esta evaluación sea especialmente relevante ya que aprovecha las descripciones propias del dataset.

Finalmente, se utilizan los pesos entrenados en el bloque anterior, por lo que no se lanza un nuevo entrenamiento, sino que se lanza únicamente la evaluación. Los resultados se añaden al mismo CSV para poder comparar los resultados con los obtenidos en el resto de experimentos.

In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"

dataset = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\refcocog"
weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam3.pt"
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"
output_weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_finetuned\\pesos_sam3_refcocog.pt"
refs_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\refcocog\\refs(umd).p"

def evaluate_model_text(dataset, weights_path, refs_path):
    sam3_wrapper = SAM(weights)
    sam3 = sam3_wrapper.model
    for param in sam3.parameters():
        param.data = param.data.to(device)
    for buffer in sam3.buffers():
        buffer.data = buffer.data.to(device)
    sd = torch.load(weights_path)["model"]
    sam3.load_state_dict(sd)
    sam3.eval()

    with open(refs_path, "rb") as f:
        refs = pickle.load(f)
    coco = COCO(os.path.join(dataset, "instances.json"))
    refs_test = [r for r in refs if r["split"] == "test"]

    iou_scores, precision_scores, recall_scores, f1_scores = [], [], [], []
    dice_scores, specificity_scores, f2_scores, pixel_accuracy_scores = [], [], [], []
    boundary_iou_scores, hausdorff_95_scores, latency_scores, vram_scores = [], [], [], []

    images_dir = os.path.join(dataset, "train2014")

    for ref in refs_test:
        ann = coco.loadAnns(ref["ann_id"])[0]
        img_info = coco.loadImgs(ref["image_id"])[0]
        img_path = os.path.join(images_dir, img_info["file_name"])

        gt_mask = coco.annToMask(ann).astype(bool)
        if gt_mask.sum() == 0:
            continue

        text_prompt = ref["sentences"][0]["raw"]

        results, latency, vram = measure_inference_sam3_prompt_refcocog(sam3_wrapper, img_path, text_prompt)

        if results is None or results[0].masks is None:
            continue

        scores = results[0].boxes.conf.cpu().numpy() if results[0].boxes is not None and len(results[0].boxes.conf) > 0 else None
        best_idx = np.argmax(scores) if scores is not None and len(scores) > 0 else 0

        H, W = gt_mask.shape
        pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        recall_scores.append(recall)
        f1_scores.append(f1)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    results = {
        "modelo": ["sam_3_refcocog_text"],
        "mean_iou": [np.mean(iou_scores)],
        "mean_f1": [np.mean(f1_scores)],
        "mean_recall": [np.mean(recall_scores)],
        "mean_precision": [np.mean(precision_scores)],
        "mean_dice": [np.mean(dice_scores)],
        "mean_specificity": [np.mean(specificity_scores)],
        "mean_f2": [np.mean(f2_scores)],
        "mean_pixel_accuracy": [np.mean(pixel_accuracy_scores)],
        "mean_boundary_iou": [np.mean(boundary_iou_scores)],
        "mean_hausdorff_95": [np.mean(hausdorff_95_scores)],
        "mean_latency_ms": [np.mean(latency_scores)],
        "mean_vram_mb": [np.mean(vram_scores)]
    }

    return results
         

start_eval = time.time()
results = evaluate_model(dataset, output_weights, refs_path)
eval_time = time.time() - start_eval

results["train_time_minutes"] = 0
results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"

if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)


Ultralytics 8.4.26  Python-3.10.11 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3090, 24575MiB)



WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]


image 1/1 C:\Users\DanielTalavera\Desktop\Trabajo_Fin_de_Grado\refcocog\images\test\COCO_train2014_000000003325.jpg: 1036x1036 1 0, 399.5ms
Speed: 7.4ms preprocess, 399.5ms inference, 1.1ms postprocess per image at shape (1, 3, 1036, 1036)

WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
image 1/1 C:\Users\DanielTalavera\Desktop\Trabajo_Fin_de_Grado\refcocog\images\test\COCO_train2014_000000004080.jpg: 1036x1036 1 0, 399.4ms
Speed: 6.2ms preprocess, 399.4ms inference, 1.1ms postprocess per image at shape (1, 3, 1036, 1036)

WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
image 1/1 C:\Users\DanielTalavera\Desktop\Trabajo_Fin_de_Grado\refcocog\images\test\COCO_train2014_000000008074.jpg: 1036x1036 1 0, 403.2ms
Speed: 5.7ms preprocess, 403.2ms inference, 1.1ms postprocess per image at shape (1, 3, 1036, 1036)

WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
image 1/1 C:\Users\DanielTalavera\Desktop\Trabajo_Fi